# 2026 FIFA World Cup Group Stage Predictions

This notebook uses our optimized feature engineering pipeline (dynamic Elo ratings + match context) and a tuned Gradient Boosting Classifier to predict the outcomes and probabilities of all 70 group stage matches for the 2026 FIFA World Cup. The dataset is fetched dynamically from Kaggle using `kagglehub`.

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
import kagglehub
import os

In [13]:
teams = [
    "Mexico", "South Africa", "South Korea", "Czech Republic",
    "Canada", "Switzerland", "Qatar", "Bosnia and Herzegovina",
    "Brazil", "Morocco", "Haiti", "Scotland",
    "United States", "Paraguay", "Australia", "Turkey",
    "Germany", "Curaçao", "Ivory Coast", "Ecuador",
    "Netherlands", "Japan", "Sweden", "Tunisia",
    "Belgium", "Egypt", "Iran", "New Zealand",
    "Spain", "Cape Verde", "Saudi Arabia", "Uruguay",
    "France", "Senegal", "Norway", "Iraq",
    "Argentina", "Algeria", "Austria", "Jordan",
    "Portugal", "DR Congo", "Uzbekistan", "Colombia",
    "England", "Croatia", "Ghana", "Panama",
]

# Download latest version of dataset from Kaggle
path = kagglehub.dataset_download("martj42/international-football-results-from-1872-to-2017")
results_path = os.path.join(path, "results.csv")

# Load raw matches from 1872 onwards
results = pd.read_csv(results_path)
results["date"] = pd.to_datetime(results["date"])
results = results.sort_values("date").reset_index(drop=True)

# Separate historical games (with scores) and future games (with NaN scores)
future_mask = results["home_score"].isnull() | results["away_score"].isnull()
historical_games = results[~future_mask].copy()
future_games = results[future_mask].copy()

# Filter future games to target World Cup matches
future_wc_games = future_games[
    future_games["tournament"] == "FIFA World Cup"
].copy()

print(f"Loaded {len(historical_games)} historical matches and {len(future_wc_games)} upcoming World Cup matches.")

Loaded 49407 historical matches and 70 upcoming World Cup matches.


In [14]:
# Compute Elo ratings on the full historical dataset to get mature ratings
elo_ratings = {}
elo_a_list = []
elo_b_list = []

k_factor = 12
home_advantage = 200

for idx, row in historical_games.iterrows():
    team_a = row["home_team"]
    team_b = row["away_team"]
    
    if team_a not in elo_ratings:
        elo_ratings[team_a] = 1500.0
    if team_b not in elo_ratings:
        elo_ratings[team_b] = 1500.0
        
    r_a = elo_ratings[team_a]
    r_b = elo_ratings[team_b]
    
    elo_a_list.append(r_a)
    elo_b_list.append(r_b)
    
    h = home_advantage if not row["neutral"] else 0.0
    expected_a = 1.0 / (1.0 + 10.0 ** ((r_b - (r_a + h)) / 400.0))
    expected_b = 1.0 - expected_a
    
    score_a = row["home_score"]
    score_b = row["away_score"]
    
    if score_a > score_b:
        actual_a, actual_b = 1.0, 0.0
    elif score_a < score_b:
        actual_a, actual_b = 0.0, 1.0
    else:
        actual_a, actual_b = 0.5, 0.5
        
    elo_ratings[team_a] = r_a + k_factor * (actual_a - expected_a)
    elo_ratings[team_b] = r_b + k_factor * (actual_b - expected_b)
    
historical_games["elo_a"] = elo_a_list
historical_games["elo_b"] = elo_b_list
historical_games["elo_diff"] = historical_games["elo_a"] - historical_games["elo_b"]

h_series = np.where(historical_games["neutral"], 0.0, home_advantage)
historical_games["elo_prob_a"] = 1.0 / (1.0 + 10.0 ** ((historical_games["elo_b"].astype(float) - (historical_games["elo_a"].astype(float) + h_series)) / 400.0))
historical_games["is_friendly"] = (historical_games["tournament"] == "Friendly").astype(int)

# Code outcomes (0 = Team A Win, 1 = Draw, 2 = Team B Win)
def get_result_code(row):
    if row["home_score"] > row["away_score"]:
        return 0
    elif row["home_score"] == row["away_score"]:
        return 1
    else:
        return 2
        
historical_games["result"] = historical_games.apply(get_result_code, axis=1)

In [15]:
# Train set preparation (from 2011 onwards)
train_data = historical_games[
    (historical_games["home_team"].isin(teams)) &
    (historical_games["away_team"].isin(teams)) &
    (historical_games["date"] >= "2011-01-01")
].copy()

features = ["elo_diff", "elo_prob_a", "neutral", "is_friendly"]
X_train = train_data[features]
y_train = train_data["result"]

# Fit our tuned Gradient Boosting model
model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
model.fit(X_train, y_train)

,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.05
,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (i

In [16]:
# Prepare future World Cup games for prediction
future_wc_features = []

for idx, row in future_wc_games.iterrows():
    team_a = row["home_team"]
    team_b = row["away_team"]
    
    # Look up current Elo (as of end of historical data)
    r_a = elo_ratings.get(team_a, 1500.0)
    r_b = elo_ratings.get(team_b, 1500.0)
    
    elo_diff = r_a - r_b
    h = home_advantage if not row["neutral"] else 0.0
    elo_prob_a = 1.0 / (1.0 + 10.0 ** ((r_b - (r_a + h)) / 400.0))
    is_friendly = 0 # World Cup games are not friendlies
    
    future_wc_features.append({
        "elo_diff": elo_diff,
        "elo_prob_a": elo_prob_a,
        "neutral": int(row["neutral"]),
        "is_friendly": is_friendly
    })
    
df_future = pd.DataFrame(future_wc_features)

# Make predictions and get probabilities
probs = model.predict_proba(df_future[features])
preds = model.predict(df_future[features])

future_wc_games["prob_a_win"] = probs[:, 0]
future_wc_games["prob_draw"] = probs[:, 1]
future_wc_games["prob_b_win"] = probs[:, 2]
future_wc_games["predicted_result"] = preds

def format_predicted_outcome(row):
    team_a = row["home_team"]
    team_b = row["away_team"]
    res = row["predicted_result"]
    if res == 0:
        return f"{team_a} Win"
    elif res == 2:
        return f"{team_b} Win"
    else:
        return "Draw"
        
future_wc_games["outcome"] = future_wc_games.apply(format_predicted_outcome, axis=1)

# Display predictions
predictions_summary = future_wc_games[[
    "date", "home_team", "away_team", "outcome", "prob_a_win", "prob_draw", "prob_b_win", "city", "country"
]].copy()

predictions_summary

,date,home_team,away_team,outcome,prob_a_win,prob_draw,prob_b_win,city,country
49407,2026-06-12,Canada,Bosnia and Herzegovina,Canada Win,0.624943,0.229990,0.145066,Toronto,Canada
49408,2026-06-12,United States,Paraguay,United States Win,0.367673,0.308076,0.324251,Inglewood,United States
49409,2026-06-13,Qatar,Switzerland,Switzerland Win,0.110350,0.206442,0.683208,Santa Clara,United States
49410,2026-06-13,Brazil,Morocco,Brazil Win,0.693354,0.183074,0.123571,East Rutherford,United States
49411,2026-06-13,Haiti,Scotland,Scotland Win,0.253839,0.304275,0.441885,Foxborough,United States
...,...,...,...,...,...,...,...,...,...
49472,2026-06-27,Colombia,Portugal,Colombia Win,0.399434,0.244190,0.356376,Miami Gardens,United States
49473,2026-06-27,Panama,England,England Win,0.237837,0.271935,0.490228,East Rutherford,United States
49474,2026-06-27,Algeria,Austria,Algeria Win,0.371274,0.309613,0.319113,Kansas City,United States
49475,2026-06-27,Jordan,Argentina,Argentina Win,0.036031,0.041078,0.922891,Arlington,United States
